# 17. 3D Face Alignment - 3ddfa_v2_mobilnet0.5_120x120

| 항목 | 내용 |
|---|---|
| 대상 모델 | `3ddfa_v2_mobilnet0.5_120x120.dxnn` |
| 카테고리 | **3D Face Alignment** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 모델이 좌표가 아니라 계수를 내놓는 구조를 설명할 수 있다.
2. 계수를 좌표로 바꾸는 데 필요한 재료가 모델 밖에 있다는 것과 그 뜻을 설명할 수 있다.
3. 값 되돌리기와 위아래 뒤집기를 빠뜨리면 무엇이 어떻게 망가지는지 설명할 수 있다.
4. 결과의 각 값이 무엇을 담는지 실측으로 확인할 수 있다.

## 1. 3D Face Alignment (얼굴 정렬)

**얼굴 사진을 받아 얼굴 부위 68곳의 위치와 얼굴이 향한 방향을 답하는 작업**

얼굴을 찾는 일이 아니다. 이미 얼굴이 중심에 잡힌 사진을 받는다. 눈·코·입·윤곽의 위치를 알면 얼굴을 똑바로
세우거나, 표정을 읽거나, 다른 그림을 얼굴에 맞춰 얹을 수 있다.

**출력의 성격**

**좌표가 아니다.** 이 모델이 내놓는 것은 숫자 62개이며, 좌표는 받는 쪽이 미리 준비된 얼굴 모형과 함께 계산해
만든다. 17개 모델 중 **모델 출력에서 최종 결과까지의 거리가 가장 먼 사례**다.

**활용 사례**

- 얼굴을 똑바로 세워 인식 앞단으로 넘기기
- 표정·시선 분석
- 얼굴에 맞춰 그림을 얹는 화면 효과

**포함되지 않는 작업**

- 얼굴을 찾기 (Face Detection)
- 누구인지 알아내기 (Face Embedding)
- 얼굴의 성별·나이 등 속성 판별 (Face Attribute)

## 2. 전체 파이프라인

```text
img/sample_face_a1.jpg  507×713 BGR
  → 전처리         BGR→RGB · 비율 무시 직접 축소 → 120×120 uint8
  → run()          입력 Tensor 하나를 전달
  → npu_0          output  [1, 62]  float32  (Public Output — NPU Task 하나뿐)
  → 후처리         값 되돌리기 → 얼굴 조립 → 자세 적용 → y 뒤집기 → 원본 크기 복원
  → Result         FaceAlignmentResult (landmarks_2d 68×2 · pose 3 · params 62)
  → 시각화         부위별로 선을 잇고 68개 점을 찍음. 자세 값을 글자로 적음
```

계수를 좌표로 바꾸는 재료(**평균 얼굴 · 변형 방향 50가지 · 되돌리기 상수**)는 모델 파일이 아니라 프로그램이
들고 있다. 이 재료가 없으면 결과를 하나도 만들 수 없다.

![17. 3D Face Alignment 전체 파이프라인](resources/17_3D_Face_Alignment_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input` |
| Shape / Dtype | `[1, 120, 120, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **직접 축소 — 비율을 지키지 않음** |
| Normalization | 없음 |

**17개 중 입력이 가장 작다.** 120×120 은 검출 계열의 640 이나 1024 와 비교가 안 된다. 연산량도 가장 적다.
그러면서 하는 일은 더 복잡해 보이는데, 이유는 **모델이 좌표를 만들지 않아서**다.

**비율을 지키지 않아도 좌표가 맞는다.** 507×713 을 120×120 으로 만들면 가로로 눌린다(배율 `0.2367` 대 `0.1683`).
검출 계열이 Letterbox 를 쓴 것과 다른데, 되돌릴 때 **축마다 다른 배율**(`4.225` 와 `5.942`)을 쓰므로 최종 좌표는
원본 위에서 제자리에 온다.

### 3.2 출력

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `output` | `[1, 62]` / `float32` | Host 후처리 | NPU Task 출력 (= Public 출력) |

**CPU Task 가 없다.** 앞의 여러 모델은 격자 출력을 정리하는 CPU Task 를 갖고 있었지만 이 모델은 NPU 하나로
끝난다. 출력 크기도 248 바이트뿐이다.

**62개 숫자의 구간**

| index | 개수 | 뜻 |
|---:|---:|---|
| 0~11 | 12 | 얼굴이 어느 방향을 보고 어디에 있는가 (3×4 행렬) |
| 12~51 | 40 | 이 사람 얼굴이 어떻게 생겼는가 (골격) |
| 52~61 | 10 | 지금 어떤 표정인가 |

좌표는 한 개도 들어 있지 않다. 이번 실행의 출력 범위는 `-2.9821` ~ `1.4742` 로, 화소 좌표라고 볼 수 없는 값이다.

**얼굴 모형 — 미리 만들어 둔 재료를 섞는다**

프로그램이 들고 있는 재료는 넷이다.

| 재료 | shape | 뜻 |
|---|---|---|
| 평균 얼굴 | `(3, 68)` | 모든 사람의 얼굴을 평균 낸 기준 얼굴 |
| 골격 변형 방향 | `(3, 68, 40)` | "코가 높아지는 방향" 같은 변형 40가지 |
| 표정 변형 방향 | `(3, 68, 10)` | "입이 벌어지는 방향" 같은 변형 10가지 |
| 되돌리기 상수 | `(62,)` × 2 | 모델이 눌러 놓은 값을 원래 크기로 되돌리는 평균·표준편차 |

```text
이 사람의 얼굴 = 평균 얼굴
               + (골격 방향 40가지 × 모델이 낸 40개 값)
               + (표정 방향 10가지 × 모델이 낸 10개 값)

사진 위 위치   = 위에서 만든 얼굴을 앞 12개 값으로 돌리고 옮긴 것
```

**공용 재료를 만들어 두고 섞는 비율만 예측하는 것**은 15번 Instance Segmentation 의 밑그림 섞기와 같은 구조다.
거기서는 밑그림 32장에 계수 32개를 곱했고, 여기서는 변형 방향 50가지에 계수 50개를 곱한다. 특정 태스크의 요령이
아니라 일반적인 설계 수법이다.

**이 재료는 모델 파일에 없다.** `.dxnn` 만 챙겨 배포하면 숫자 62개를 좌표로 바꿀 방법이 없어 **아무 결과도 나오지
않는다.** 01번의 부류 이름 표와 비슷한 구도지만 훨씬 무겁다 — 이름표가 없어도 번호는 나오지만, 얼굴 모형이 없으면
계산 자체가 불가능하다.

**앞 12개는 회전만이 아니다.** 3×4 행렬의 왼쪽 3×3 을 회전으로, 마지막 열을 이동으로 쓰는데, 이 3×3 에는 **크기
배율까지 섞여 있다.** 그래서 얼굴 모형의 수만 단위 좌표가 곱셈 한 번에 120 안쪽으로 들어온다. 이번 실행에서 행
길이는 `0.00061`·`0.00057`·`0.00061` 로, 순수 회전이라면 1 이어야 하는 값이다.

**좌표에는 이 배율이 필요하다.** 배율이 있어야 좌표 68곳이 입력 크기 안에 들어온다. 반대로 **각도를 쓸 때는
걷어낸다** — 배율이 남아 있으면 세 각도 중 하나가 그만큼 작아진 채로 나온다. 4.4 에서 그 구조를 보고 4.5 에서
각 값이 무엇을 담는지 확인한다.

#### 68개 점의 번호와 부위

이 모델은 점의 이름 배열을 갖고 있지 않다. 68곳의 배치는 iBUG 가 정한 것을 따르며, 아래 그림이 그 배치다.

![iBUG 68 facial points](resources/17_3D_Face_Alignment_landmarks.jpg)

출처: [iBUG, Imperial College London — Facial point annotations](https://ibug.doc.ic.ac.uk/resources/facial-point-annotations/)

**그림은 1~68 로 세고 이 노트북의 배열은 0~67 로 센다.** 한 칸 차이가 있으므로 그림의 번호를 그대로 배열 index
로 쓰면 안 된다.

| 부위 | 배열 index | 점 수 | 그림 번호 |
|---|---|---:|---|
| 얼굴 윤곽 | 0~16 | 17 | 1~17 |
| 왼쪽 눈썹 | 17~21 | 5 | 18~22 |
| 오른쪽 눈썹 | 22~26 | 5 | 23~27 |
| 콧대 | 27~30 | 4 | 28~31 |
| 코 아래 | 31~35 | 5 | 32~36 |
| 왼쪽 눈 | 36~41 | 6 | 37~42 |
| 오른쪽 눈 | 42~47 | 6 | 43~48 |
| 입 바깥 | 48~59 | 12 | 49~60 |
| 입 안 | 60~67 | 8 | 61~68 |

왼쪽·오른쪽은 **보는 사람 기준**이다. 이번 실행에서 index 36~41 의 x 평균이 `165`, index 42~47 이 `329` 로,
앞 묶음이 화면 왼쪽에 있다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. Task 가 `npu_0` 하나뿐이고
출력이 `[1, 62]` 248 바이트인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "3ddfa_v2_mobilnet0.5_120x120.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `top_k` | 5 | — | **쓰이지 않음** |

`top_k` 는 후처리 어디에서도 읽지 않는다. 상위 몇 개를 고르는 개념이 없는 태스크이므로 분류 모델의 설정이 그대로
복사된 흔적이다. **설정 파일에 값이 있다는 것이 그 값이 동작을 바꾼다는 뜻은 아니다.**

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고, 얼굴 모형 재료를 적재한다. 이후 셀은 여기서 만든 이름을 그대로
쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_face_a1.jpg"
CONFIG_PATH = (DX_APP / "src" / "python_example" / "face_alignment"
               / "3ddfa_v2_mobilnet0_5_120x120" / "config.json")
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.processors._bfm_data import load_bfm  # noqa: E402

# 얼굴 모형 재료. 노트북이 스스로 만들어 낼 수 없으므로 원본이 들고 있는 것을 그대로 쓴다.
BFM = load_bfm()

print("config", CONFIG)
for name, array in BFM.items():
    print(f"  {name:12s} {str(array.shape):14s} {array.dtype} "
          f"min={array.min():12.4f} max={array.max():12.4f}")

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 크기가 좌표 복원의 근거다
- `FaceAlignmentResult` — 후처리가 만들어 시각화로 넘기는 결과. 좌표 68곳·자세 3개·원본 계수 62개를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0


@dataclass
class FaceAlignmentResult:
    """얼굴 68곳의 좌표와 얼굴이 향한 방향. 원본 대응: common/base/i_processor.py"""

    landmarks_2d: np.ndarray = None
    landmarks_3d: np.ndarray = None
    pose: list = field(default_factory=list)
    params: np.ndarray = None


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):16s} {array.dtype!s:8s} "
          f"min={array.min():12.4f} max={array.max():12.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (6, 8)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
print("input :", engine.get_input_tensors_info())
print("output:", engine.get_output_tensors_info())

INPUT_HEIGHT, INPUT_WIDTH = engine.get_input_tensors_info()[0]["shape"][1:3]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

색 순서를 바꾸고 120×120 으로 직접 줄인다. 비율을 지키지 않으므로 얼굴이 가로로 눌린다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 120×120, 비율 무시
- dtype — `uint8` 유지 (나누기 없음)
- Context — 원본 크기만 넘긴다 (여백이 없으므로 pad 가 필요 없다)

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 얼굴이 이미 화면을 채우고 있다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(5, 7))

#### 전처리 함수 정의

- `face_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `face_preprocessor_process` — 색 변환과 직접 축소를 적용하고 원본 크기를 Context 에 담음

In [ ]:
def face_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def face_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (state["input_width"], state["input_height"]),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 가로와 세로의 배율이 다른 것을 확인한다.

In [ ]:
preprocessor_state = face_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = face_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"input={ctx.input_width}x{ctx.input_height}")
print(f"축소 배율     가로 {ctx.input_width / ctx.original_width:.4f} "
      f"세로 {ctx.input_height / ctx.original_height:.4f} — 다르므로 얼굴이 가로로 눌린다")

notebook_show_bgr(cv2.cvtColor(input_tensor, cv2.COLOR_RGB2BGR),
                  f"Resized {INPUT_WIDTH}x{INPUT_HEIGHT} (aspect not kept)", figsize=(3, 3))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이 포함돼
있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
for index, output in enumerate(outputs):
    notebook_describe_array(f"output[{index}]", output)

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 개수 — 62개
- 값의 범위 — 화소 좌표로 볼 수 없는 크기다
- 구간 — 앞 12·다음 40·마지막 10 이 각각 무엇인지

In [ ]:
raw_params = np.asarray(outputs[0]).flatten().astype(np.float32)
notebook_describe_array("raw params", raw_params)

print("앞 12개 (자세):", np.array2string(raw_params[:12], precision=4))
print("골격 40개 범위:", f"{raw_params[12:52].min():.4f} ~ {raw_params[12:52].max():.4f}")
print("표정 10개 범위:", f"{raw_params[52:62].min():.4f} ~ {raw_params[52:62].max():.4f}")
print("\n이 62개 안에 좌표는 없다. 좌표는 4.4 에서 얼굴 모형과 함께 계산해 만든다.")

### 4.4 후처리

다섯 단계다. **각 단계를 빠뜨렸을 때 나는 증상이 다르며, 어느 것도 에러를 내지 않는다.**

| 단계 | 하는 일 | 빠뜨리면 |
|---|---|---|
| ① 값 되돌리기 | `raw × 표준편차 + 평균` | 좌표가 수십만 단위로 튀어 사진 밖으로 나간다 |
| ② 얼굴 조립 | 평균 얼굴 + 골격 변형 + 표정 변형 | 누구를 넣어도 똑같은 평균 얼굴이 나온다 |
| ③ 자세 적용 | 3×3 을 곱하고 마지막 열을 더함 | 좌표가 얼굴 모형의 수만 단위로 남는다 |
| ④ y 뒤집기 | `y ← 입력 높이 - y` | 얼굴이 위아래로 뒤집힌다 |
| ⑤ 크기 복원 | 축마다 다른 배율을 곱함 | 좌표가 왼쪽 위 작은 구석에 몰린다 |

① 의 상수도 얼굴 모형 재료에 함께 들어 있다. 모델이 학습 편의를 위해 값을 눌러 놓았기 때문에 필요한 단계다.

**자세 값은 3×3 에서 뽑는다.** 세 각도를 역삼각함수로 얻는데, 계산 방식이 둘로 갈린다.

| 계산 | 배율이 섞인 경우 |
|---|---|
| 행렬 값 **두 개의 비**를 씀 (`atan2`) | 분자·분모에서 약분돼 그대로 나옴 |
| 행렬 값 **하나**를 그대로 씀 (`asin`) | 배율을 곱한 만큼 작아진 값이 나옴 |

원본은 배율을 걷어내지 않으므로 `asin` 으로 구하는 각도가 0 에 가까운 값으로 나온다. **그 값이 어느 방향의
각도인지는 4.5 에서 사진으로 확인한다.**

#### 후처리 함수 정의

- `face_postprocessor_create` — 얼굴 모형 재료와 입력 크기를 state 에 담음
- `face_extract_pose` — 3×3 에서 yaw·pitch·roll 을 뽑음
- `face_reconstruct_landmarks` — 얼굴 조립·자세 적용·y 뒤집기·크기 복원
- `face_postprocessor_process` — 값 되돌리기부터 Result 생성까지

In [ ]:
def face_postprocessor_create(bfm: dict, input_width: int, input_height: int) -> dict:
    """얼굴 모형 재료와 입력 크기를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"bfm": bfm, "input_width": input_width, "input_height": input_height}


def face_extract_pose(rotation: np.ndarray) -> list:
    """3×3 에서 yaw·pitch·roll 을 도(degree)로 뽑는다. 원본 대응: TDDFAPostprocessor._extract_pose()"""
    sin_pitch = float(np.clip(rotation[2, 0], -1.0, 1.0))
    pitch = np.arcsin(sin_pitch)
    cos_pitch = np.cos(pitch)
    if abs(cos_pitch) > 1e-6:
        yaw = np.arctan2(rotation[2, 1] / cos_pitch, rotation[2, 2] / cos_pitch)
        roll = np.arctan2(rotation[1, 0] / cos_pitch, rotation[0, 0] / cos_pitch)
    else:
        yaw = 0.0
        roll = np.arctan2(-rotation[0, 1], rotation[1, 1])
    return [float(np.degrees(yaw)), float(np.degrees(pitch)), float(np.degrees(roll))]


def face_reconstruct_landmarks(state: dict, rotation: np.ndarray, offset: np.ndarray,
                               alpha_shape: np.ndarray, alpha_expression: np.ndarray,
                               ctx: PreprocessContext) -> np.ndarray:
    """계수를 68곳의 좌표로 바꾼다. 원본 대응: TDDFAPostprocessor._reconstruct_landmarks()"""
    bfm = state["bfm"]

    # ② 조립 — 평균 얼굴에 골격·표정 변형을 더한다. 결과는 (3, 68)
    shape_deform = np.einsum("ijk,kl->ij", bfm["w_shp_base"], alpha_shape)
    expression_deform = np.einsum("ijk,kl->ij", bfm["w_exp_base"], alpha_expression)
    vertices = bfm["u_base"] + shape_deform + expression_deform

    # ③ 자세 — 3×3 에 배율이 섞여 있어 이 곱셈 한 번에 좌표가 입력 크기 안쪽으로 들어온다.
    points = rotation @ vertices + offset

    # ④ 얼굴 모형은 위쪽이 y 증가, 사진은 아래쪽이 y 증가다. 뒤집지 않으면 얼굴이 거꾸로 나온다.
    points[0, :] -= 1
    points[1, :] = state["input_height"] - points[1, :]

    # ⑤ 축마다 다른 배율을 곱해 원본 크기로 되돌린다.
    scale_x = ctx.original_width / state["input_width"]
    scale_y = ctx.original_height / state["input_height"]
    landmarks = np.zeros((68, 2), dtype=np.float32)
    landmarks[:, 0] = points[0, :] * scale_x
    landmarks[:, 1] = points[1, :] * scale_y
    return landmarks


def face_postprocessor_process(state: dict, outputs: list,
                               ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    if not outputs:
        return []

    raw = np.asarray(outputs[0]).flatten().astype(np.float32)
    # ① 되돌리기 — 상수는 얼굴 모형 재료에 함께 들어 있다.
    params = raw * state["bfm"]["param_std"] + state["bfm"]["param_mean"]

    matrix = params[:12].reshape(3, 4)
    rotation = matrix[:, :3]
    offset = matrix[:, 3:].reshape(3, 1)
    alpha_shape = params[12:52].reshape(-1, 1)
    alpha_expression = params[52:62].reshape(-1, 1)

    result = FaceAlignmentResult()
    result.params = raw
    result.pose = face_extract_pose(rotation)
    result.landmarks_2d = face_reconstruct_landmarks(state, rotation, offset, alpha_shape,
                                                     alpha_expression, ctx)
    # 세 번째 축은 계산된 깊이가 아니라 0 으로 채운 자리다.
    result.landmarks_3d = np.column_stack([result.landmarks_2d,
                                           np.zeros(len(result.landmarks_2d))])
    return [result]

#### 값 되돌리기 단계 확인

되돌리기가 값의 크기를 얼마나 바꾸는지, 배율이 섞인 3×3 이 어떤 모양인지 확인한다.

In [ ]:
postprocessor_state = face_postprocessor_create(BFM, INPUT_WIDTH, INPUT_HEIGHT)
denormalized = raw_params * BFM["param_std"] + BFM["param_mean"]

notebook_describe_array("raw", raw_params)
notebook_describe_array("되돌린 값", denormalized)

pose_matrix = denormalized[:12].reshape(3, 4)
print("\n3×3 (회전 + 배율):")
print(np.array2string(pose_matrix[:, :3], precision=6))
print("마지막 열 (이동):", np.array2string(pose_matrix[:, 3], precision=3))
print("각 행의 길이:", np.array2string(np.linalg.norm(pose_matrix[:, :3], axis=1), precision=6),
      "— 순수 회전이면 1 이어야 한다")

#### 조립 단계 확인

평균 얼굴에 골격·표정 변형이 얼마씩 더해지는지 확인한다.

In [ ]:
alpha_shape = denormalized[12:52].reshape(-1, 1)
alpha_expression = denormalized[52:62].reshape(-1, 1)
shape_deform = np.einsum("ijk,kl->ij", BFM["w_shp_base"], alpha_shape)
expression_deform = np.einsum("ijk,kl->ij", BFM["w_exp_base"], alpha_expression)
vertices = BFM["u_base"] + shape_deform + expression_deform

notebook_describe_array("평균 얼굴", BFM["u_base"])
notebook_describe_array("골격 변형분", shape_deform)
notebook_describe_array("표정 변형분", expression_deform)
notebook_describe_array("조립된 얼굴", vertices)
print(f"\n변형분의 절대 크기 평균 — 골격 {np.abs(shape_deform).mean():.2f} "
      f"표정 {np.abs(expression_deform).mean():.2f}")
span = float(BFM["u_base"].max() - BFM["u_base"].min())
print(f"평균 얼굴의 좌표 범위 {span:,.0f} 에 대해 변형분은 "
      f"{np.abs(shape_deform + expression_deform).mean() / span * 100:.1f}% — "
      "얼굴 모양의 대부분은 평균 얼굴이 이미 설명하고, 계수는 개인차만 얹는다")

#### 후처리 실행

확인할 것:

- 좌표 개수와 범위 — 원본 사진 안에 들어오는지
- 자세 — yaw·pitch·roll 이 몇 도인지
- 부위별 index 구간 — 실행 좌표의 평균 위치가 부위와 맞는지

In [ ]:
results = face_postprocessor_process(postprocessor_state, outputs, ctx)
result = results[0]

notebook_describe_array("landmarks_2d", result.landmarks_2d)
print(f"원본 사진 크기 {ctx.original_width}x{ctx.original_height} 안에 들어온다")
print(f"pose  yaw={result.pose[0]:.2f}  pitch={result.pose[1]:.4f}  roll={result.pose[2]:.2f}  (도)")

FACE_PARTS = {
    "contour": list(range(0, 17)), "left_eyebrow": list(range(17, 22)),
    "right_eyebrow": list(range(22, 27)), "nose_bridge": list(range(27, 31)),
    "nose_bottom": list(range(31, 36)), "left_eye": list(range(36, 42)) + [36],
    "right_eye": list(range(42, 48)) + [42], "outer_lip": list(range(48, 60)) + [48],
    "inner_lip": list(range(60, 68)) + [60],
}
print("\n부위별 평균 위치:")
for part_name, indices in FACE_PARTS.items():
    unique = sorted(set(indices))
    center = result.landmarks_2d[unique].mean(axis=0)
    print(f"  {part_name:14s} index {unique[0]:2d}~{unique[-1]:2d} ({len(unique):2d}점) "
          f"평균=({center[0]:6.1f}, {center[1]:6.1f})")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 부위별로 색을 달리해 선을 잇고 68개 점을 찍는다. 눈과 입술은 처음 index 를 뒤에 한 번 더 붙여
고리를 닫는다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

왼쪽 위에 찍히는 `Yaw`·`Pitch` 글자는 원본이 붙인 이름 그대로다. 각 값이 담는 방향은 이 절 마지막에서 사진으로
확인한다.

In [ ]:
PART_COLORS = {
    "contour": (200, 200, 200), "left_eyebrow": (0, 255, 0), "right_eyebrow": (0, 255, 0),
    "nose_bridge": (255, 200, 0), "nose_bottom": (255, 200, 0),
    "left_eye": (255, 0, 0), "right_eye": (255, 0, 0),
    "outer_lip": (0, 0, 255), "inner_lip": (0, 100, 255),
}


def face_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()
    if not results:
        return output

    landmarks = results[0].landmarks_2d
    pose = results[0].pose

    for part_name, indices in FACE_PARTS.items():
        color = PART_COLORS.get(part_name, (255, 255, 255))
        for position in range(len(indices) - 1):
            first, second = indices[position], indices[position + 1]
            if first < len(landmarks) and second < len(landmarks):
                cv2.line(output, (int(landmarks[first, 0]), int(landmarks[first, 1])),
                         (int(landmarks[second, 0]), int(landmarks[second, 1])),
                         color, 1, cv2.LINE_AA)

    for index in range(min(68, len(landmarks))):
        cv2.circle(output, (int(landmarks[index, 0]), int(landmarks[index, 1])), 2,
                   (0, 255, 255), -1)

    if pose and len(pose) >= 3:
        cv2.putText(output, f"Yaw: {pose[0]:.1f} Pitch: {pose[1]:.1f} Roll: {pose[2]:.1f}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return output

#### 결과 시각화

68곳의 좌표와 자세 값을 inline 으로 표시한다.

In [ ]:
output_image = face_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "68 facial landmarks", figsize=(5, 7))

#### 뒤집기와 되돌리기를 빠뜨리면

두 단계를 각각 빼고 같은 계수로 좌표를 다시 만들어 본다. 둘 다 **에러 없이 그림만 이상해지므로** 눈으로 확인하는
수밖에 없다. 되돌리기를 뺀 쪽은 좌표가 사진 밖으로 나가 화면에 선이 하나도 남지 않고, y 뒤집기를 뺀 쪽은 좌표의
범위가 정상과 똑같아 **숫자만 보면 정상과 구별되지 않는다** — 눈썹이 턱보다 아래에 오는 것으로 알아낸다.

In [ ]:
def landmarks_variant(skip_denormalize: bool = False, skip_flip: bool = False) -> np.ndarray:
    """단계를 하나 빼고 좌표를 만든다. 노트북 전용 실험."""
    params = raw_params if skip_denormalize else raw_params * BFM["param_std"] + BFM["param_mean"]
    matrix = params[:12].reshape(3, 4)
    points = (matrix[:, :3] @ (BFM["u_base"]
                               + np.einsum("ijk,kl->ij", BFM["w_shp_base"],
                                           params[12:52].reshape(-1, 1))
                               + np.einsum("ijk,kl->ij", BFM["w_exp_base"],
                                           params[52:62].reshape(-1, 1)))
              + matrix[:, 3:].reshape(3, 1))
    points[0, :] -= 1
    if not skip_flip:
        points[1, :] = INPUT_HEIGHT - points[1, :]
    return np.column_stack([points[0, :] * ctx.original_width / INPUT_WIDTH,
                            points[1, :] * ctx.original_height / INPUT_HEIGHT]).astype(np.float32)


EYEBROW = list(range(17, 27))
CHIN = 8


def report_variant(title: str, landmarks: np.ndarray) -> None:
    """좌표의 크기와 눈썹·턱의 위아래 순서를 한 줄로 출력한다. 노트북 전용."""
    span = landmarks.max(axis=0) - landmarks.min(axis=0)
    eyebrow_y = float(landmarks[EYEBROW, 1].mean())
    chin_y = float(landmarks[CHIN, 1])
    order = "눈썹이 위" if eyebrow_y < chin_y else "눈썹이 아래 (뒤집힘)"
    print(f"{title:12s} x 범위 {span[0]:9.1f}  y 범위 {span[1]:9.1f}  "
          f"눈썹 y={eyebrow_y:8.1f} 턱 y={chin_y:8.1f}  {order}")


report_variant("정상", result.landmarks_2d)
broken = []
for title, kwargs in [("되돌리기 없음", {"skip_denormalize": True}),
                      ("뒤집기 없음", {"skip_flip": True})]:
    variant = FaceAlignmentResult()
    variant.landmarks_2d = landmarks_variant(**kwargs)
    variant.pose = result.pose
    report_variant(title, variant.landmarks_2d)
    broken.append((title, face_visualizer_visualize(image, [variant])))

notebook_show_bgr(broken[0][1], "no denormalize - nothing lands on the photo", figsize=(5, 7))
notebook_show_bgr(broken[1][1], "no y-flip - face is upside down", figsize=(5, 7))

#### landmarks_3d 의 세 번째 축

`landmarks_3d` 의 세 번째 축에 무엇이 담기는지 재 본다. 배율이 어느 각도에 영향을 주는지도 함께 본다.

In [ ]:
print("landmarks_3d 세 번째 축 :",
      f"min={result.landmarks_3d[:, 2].min():.1f} max={result.landmarks_3d[:, 2].max():.1f}"
      " — 전부 0")

points_3d = pose_matrix[:, :3] @ vertices + pose_matrix[:, 3:].reshape(3, 1)
print(f"계산 과정의 깊이       : {points_3d[2].min():.1f} ~ {points_3d[2].max():.1f} "
      "— 계산은 됐지만 결과에 담기지 않는다")

scale = float((np.linalg.norm(pose_matrix[0, :3]) + np.linalg.norm(pose_matrix[1, :3])) / 2)
print(f"\n3×3 에 섞인 배율        : {scale:.6f}")
print("배율을 그대로 둔 자세  : yaw={:.2f} pitch={:.4f} roll={:.2f}".format(
    *face_extract_pose(pose_matrix[:, :3])))
print("배율을 나눈 뒤의 자세  : yaw={:.2f} pitch={:.4f} roll={:.2f}".format(
    *face_extract_pose(pose_matrix[:, :3] / scale)))
print("\n세 값 중 pitch 만 바뀐다. asin 으로 구하는 값이다.")

#### 자세 세 값이 담는 방향

세 값이 각각 어느 방향의 각도인지 **좌우로 돌아간 얼굴을 넣어** 확인한다. 옆을 본 사진에서 커지는 값이 곧
좌우 회전이다.

In [ ]:
def pose_of(image_bgr: np.ndarray) -> tuple:
    """사진 한 장의 자세를 배율 포함·제거 두 가지로 구한다. 노트북 전용 실험."""
    tensor, _ = face_preprocessor_process(preprocessor_state, image_bgr)
    values = np.asarray(engine.run([tensor])[0]).flatten().astype(np.float32)
    matrix = (values * BFM["param_std"] + BFM["param_mean"])[:12].reshape(3, 4)[:, :3]
    local_scale = float((np.linalg.norm(matrix[0]) + np.linalg.norm(matrix[1])) / 2)
    return face_extract_pose(matrix), face_extract_pose(matrix / local_scale)


turned = cv2.imread(str(DX_APP / "sample" / "img/sample_face_a2.jpg"))

print(f"{'':30s} {'yaw':>8s} {'pitch':>10s} {'roll':>8s}")
for title, photo in [("정면 (sample_face_a1)", image), ("옆을 봄 (sample_face_a2)", turned)]:
    as_is, without_scale = pose_of(photo)
    print(f"{title:30s} {as_is[0]:8.2f} {as_is[1]:10.4f} {as_is[2]:8.2f}   원본 코드 그대로")
    print(f"{'  └ 배율을 걷어내면':30s} {without_scale[0]:8.2f} {without_scale[1]:10.2f} "
          f"{without_scale[2]:8.2f}")

height = 320
pair = cv2.hconcat([cv2.resize(photo, (int(photo.shape[1] * height / photo.shape[0]), height))
                    for photo in (image, turned)])
notebook_show_bgr(pair, "Front vs turned - which value moves?", figsize=(7, 5))

옆을 본 사진에서 커진 값은 `pitch` 다. 세 값이 담는 방향은 이렇다.

| 값 | 계산 | 담는 방향 | 배율의 영향 |
|---|---|---|---|
| `yaw` | `atan2` | 위아래 | 없음 |
| `pitch` | `asin` | **좌우** | 배율만큼 작아짐 |
| `roll` | `atan2` | 기울기 | 없음 |

**좌우 회전을 쓰려면 배율을 걷어내고 `pitch` 를 읽는다.** 걷어내면 48도가 그대로 나오므로 필요한 정보는 계수
62개 안에 다 들어 있다.

값의 이름만 보고 쓰지 말고 이렇게 사진으로 대응을 확인해 두는 것이 이 모델에서 얻어 갈 습관이다.